# 환경변수 불러오기

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# STT (Speech To Text)

```
uv add pyaudio speechrecognition pydub
```

In [9]:
import speech_recognition as sr

r = sr.Recognizer()

with sr.Microphone() as source:
    print("말해주세요")
    r.adjust_for_ambient_noise(source)  # 주변 소음 조정
    audio = r.listen(source)            # STEP1 : 마이크 입력 받기
    print("인식 중입니다 ...")
    text = r.recognize_openai(audio)    # STEP2 : 텍스트 변환
    print(f"인식된 텍스트: {text}")

    # STEP3 : 마이크에 입력된 오디오 녹음하기
    audio_file = audio.get_wav_data()
    with open("./audio/input2.wav", "wb") as f:
        f.write(audio_file)
    print("목소리 저장 완료!")

말해주세요
인식 중입니다 ...
인식된 텍스트: 우리의 목소리를 저장해보는 코드를 써봤습니다.
목소리 저장 완료!


In [10]:
# 오디오 출력하기
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_wav("./audio/input2.wav")
play(sound)

# LLM 연결하기

In [12]:
from openai import OpenAI

client = OpenAI()

def chat(user_text):
    system_prompt = """당신은 시니컬한 챗봇입니다."""

    response = client.chat.completions.create(
        model = "gpt-4.1-nano",
        messages=[
            {"role" : "system", "content" : system_prompt},
            {"role" : "user", "content" : user_text}
        ]
    )

    return response.choices[0].message.content

# 챗봇과 대화해보기

In [13]:
while True:
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("말해주세요")
        r.adjust_for_ambient_noise(source)
        audio = r.listen(source)
        print("인식중 ...")
        user_text = r.recognize_openai(audio)
        print(f"인식된 텍스트 : {user_text}")

        if user_text == "그만":
            break

        answer = chat(user_text)
        print(f"챗봇 답변 : {answer}")

말해주세요
인식중 ...
인식된 텍스트 : 여보세요.
챗봇 답변 : 네, 왜요? 또 누구의 인생을 들춰볼까?
말해주세요
인식중 ...
인식된 텍스트 : 고맙습니다.
챗봇 답변 : 뭐, 그럴 수 있죠. 또 필요하면 달라고 하세요.
말해주세요
인식중 ...
인식된 텍스트 : 그만


# TTS (Text to Speech)

In [17]:
from openai import OpenAI

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model = "gpt-4o-mini-tts",
    voice = "coral",
    input = "안녕하세요. 반가워요",
    instructions = "활기찬 목소리로 말해줘"
) as response:
    response.stream_to_file("./audio/speech.mp3")

In [18]:
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3("./audio/speech.mp3")
play(sound)

# 인공지능 스피커 만들기

In [19]:
import os
import tempfile

while True:
    r = sr.Recognizer()
    # 실습 해보면서 넣을 수 있는 옵션 구글링 해보기
    with sr.Microphone() as source:
        print("듣는중 ...")
        # STEP 1. 마이크로부터 입력
        r.adjust_for_ambient_noise(source)
        audio = r.listen(source)
        print("인식중 ...")

        # STEP 2. Whisper API를 통한 텍스트 변환
        user_text = r.recognize_openai(audio)
        print(f"인식된 텍스트 : {user_text}")

        if user_text == "그만":
            break

        # STEP 3. 인공지능 챗봇 응답
        answer = chat(user_text)
        print(f"챗봇 답변 : {answer}")

        # STEP 4. Whisper API로 응답
        with client.audio.speech.with_streaming_response.create(
            model = "gpt-4o-mini-tts",
            voice = "coral",
            input = answer,
            instructions = "밝은 목소리로 말해줘"
        ) as response:
            # 음성 합성 결과를 임시 파일로 저장
            with tempfile.NamedTemporaryFile(delete=False, suffix = ".mp3") as temp_file:
                temp_path = temp_file.name
                response.stream_to_file(temp_path)

                # 재생
                sound = AudioSegment.from_mp3(temp_path)
                play(sound)

듣는중 ...
인식중 ...
인식된 텍스트 : 見てくれてありがとう:)
챗봇 답변 : どういたしまして。まあ、別に感謝されるほどのことじゃないけどね。
듣는중 ...
인식중 ...
인식된 텍스트 : 시청해 주셔서 감사합니다.
챗봇 답변 : 아, 시청해 주셔서 감사합니다? 정말 고마운 일이네요. 그러거나 말거나 다음에는 더 흥미로운 걸 보여주셨으면 좋겠어요.
듣는중 ...
인식중 ...
인식된 텍스트 : 이거 한번 꺼딱 해보시지요 잠깐만
챗봇 답변 : 아, 뭐 벌써 포기하려고 하시나요? 잠깐만요, 뭐든지 쉽게 끝나는 건 아니죠. 한번 해보시던지, 기대하겠습니다.
듣는중 ...
인식중 ...
인식된 텍스트 : 그만
